In [2]:
# Lab 8 Part C: Graph Convolutional Network (GCN)
# 100% Correct Working Code

# ==============================
# INSTALL REQUIRED LIBRARIES
# ==============================

!pip install torch-geometric -q

# ==============================
# IMPORT LIBRARIES
# ==============================

import torch
import torch.nn.functional as F
from torch_geometric.datasets import Planetoid
from torch_geometric.nn import GCNConv

# ==============================
# DEVICE CONFIGURATION
# ==============================

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("Using Device:", device)

# ==============================
# LOAD DATASET
# ==============================

dataset = Planetoid(root="data/Planetoid", name="Cora")

data = dataset[0].to(device)

print("Dataset Loaded Successfully")
print("Number of Features:", dataset.num_features)
print("Number of Classes:", dataset.num_classes)

# ==============================
# DEFINE GCN MODEL
# ==============================

class GCN(torch.nn.Module):

    def __init__(self):
        super(GCN, self).__init__()

        self.conv1 = GCNConv(dataset.num_features, 16)

        self.conv2 = GCNConv(16, dataset.num_classes)

    def forward(self, data):

        x = data.x
        edge_index = data.edge_index

        # First GCN Layer
        x = self.conv1(x, edge_index)

        x = F.relu(x)

        x = F.dropout(x, training=self.training)

        # Second GCN Layer
        x = self.conv2(x, edge_index)

        return F.log_softmax(x, dim=1)

# ==============================
# MODEL INITIALIZATION
# ==============================

model = GCN().to(device)

optimizer = torch.optim.Adam(
    model.parameters(),
    lr=0.01,
    weight_decay=5e-4
)

# ==============================
# TRAINING FUNCTION
# ==============================

def train():

    model.train()

    optimizer.zero_grad()

    out = model(data)

    loss = F.nll_loss(
        out[data.train_mask],
        data.y[data.train_mask]
    )

    loss.backward()

    optimizer.step()

    return loss.item()

# ==============================
# TEST FUNCTION
# ==============================

def test():

    model.eval()

    out = model(data)

    pred = out.argmax(dim=1)

    correct = (
        pred[data.test_mask]
        ==
        data.y[data.test_mask]
    ).sum()

    accuracy = int(correct) / int(data.test_mask.sum())

    return accuracy

# ==============================
# TRAIN MODEL
# ==============================

for epoch in range(1, 201):

    loss = train()

    if epoch % 20 == 0:

        acc = test()

        print(
            f"Epoch: {epoch:03d} | "
            f"Loss: {loss:.4f} | "
            f"Test Accuracy: {acc:.4f}"
        )

# ==============================
# FINAL ACCURACY
# ==============================

final_acc = test()

print("\nFinal Test Accuracy:", round(final_acc * 100, 2), "%")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.7/63.7 kB 1.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 12.6 MB/s eta 0:00:00
Using Device: cpu


Processing...
Done!


Dataset Loaded Successfully
Number of Features: 1433
Number of Classes: 7
Epoch: 020 | Loss: 0.2713 | Test Accuracy: 0.7910
Epoch: 040 | Loss: 0.0596 | Test Accuracy: 0.7800
Epoch: 060 | Loss: 0.0665 | Test Accuracy: 0.7820
Epoch: 080 | Loss: 0.0406 | Test Accuracy: 0.7860
Epoch: 100 | Loss: 0.0259 | Test Accuracy: 0.7890
Epoch: 120 | Loss: 0.0270 | Test Accuracy: 0.7980
Epoch: 140 | Loss: 0.0406 | Test Accuracy: 0.7970
Epoch: 160 | Loss: 0.0298 | Test Accuracy: 0.7990
Epoch: 180 | Loss: 0.0398 | Test Accuracy: 0.8020
Epoch: 200 | Loss: 0.0203 | Test Accuracy: 0.8170

Final Test Accuracy: 81.7 %
